# Egypt Weather Data ETL Prototype

This cleaned DEPI exercise retrieves **current** weather observations for ten Egyptian cities, converts the API response into a pandas DataFrame, enriches the data, and validates the result. It is an ingestion prototype—not a weather forecasting model.

The original classroom notebook embedded credentials. This public version reads `OPENWEATHER_API_KEY` from the environment and contains no SQL password.

In [ ]:
import os
from datetime import datetime, timezone

import pandas as pd
import requests

API_URL = "https://api.openweathermap.org/data/2.5/weather"
API_KEY = os.environ.get("OPENWEATHER_API_KEY")
if not API_KEY:
    raise RuntimeError("Set OPENWEATHER_API_KEY before running this notebook.")

CITIES = (
    "Cairo", "Giza", "Alexandria", "Port Said", "Suez",
    "Ismailia", "Mansoura", "Tanta", "Luxor", "Aswan",
)

## Extract
Use a session, a finite timeout, and HTTP error handling for every city request.

In [ ]:
def fetch_city_weather(session: requests.Session, city: str) -> dict:
    response = session.get(
        API_URL,
        params={"appid": API_KEY, "q": f"{city},EG", "units": "metric"},
        timeout=15,
    )
    response.raise_for_status()
    payload = response.json()
    weather = payload["weather"][0]
    return {
        "City": city,
        "Country": payload["sys"]["country"],
        "Latitude": payload["coord"]["lat"],
        "Longitude": payload["coord"]["lon"],
        "TemperatureC": payload["main"]["temp"],
        "PressureHPa": payload["main"]["pressure"],
        "HumidityPct": payload["main"]["humidity"],
        "WindSpeedMps": payload["wind"]["speed"],
        "WeatherMain": weather["main"],
        "WeatherDescription": weather["description"],
    }

with requests.Session() as session:
    records = [fetch_city_weather(session, city) for city in CITIES]

weather_df = pd.DataFrame.from_records(records)

## Transform and enrich
Map each city to a region, derive human-readable categories, and record ingestion time in UTC.

In [ ]:
REGION_BY_CITY = {
    "Cairo": "Greater Cairo", "Giza": "Greater Cairo",
    "Alexandria": "North Coast",
    "Port Said": "Canal", "Suez": "Canal", "Ismailia": "Canal",
    "Mansoura": "Lower Egypt", "Tanta": "Lower Egypt",
    "Luxor": "Upper Egypt", "Aswan": "Upper Egypt",
}

def temperature_category(value: float) -> str:
    if value < 17:
        return "Cold"
    if value < 28:
        return "Mild"
    return "Hot"

def humidity_category(value: float) -> str:
    if value < 30:
        return "Low"
    if value <= 60:
        return "Medium"
    return "High"

weather_df["Region"] = weather_df["City"].map(REGION_BY_CITY)
weather_df["TemperatureCategory"] = weather_df["TemperatureC"].map(temperature_category)
weather_df["HumidityCategory"] = weather_df["HumidityPct"].map(humidity_category)
weather_df["IngestionTimeUTC"] = datetime.now(timezone.utc)

## Validate
Fail before loading when the result is incomplete, duplicated, or outside basic physical/API ranges.

In [ ]:
required = {
    "City", "Country", "Latitude", "Longitude", "TemperatureC",
    "PressureHPa", "HumidityPct", "WindSpeedMps", "Region",
    "TemperatureCategory", "HumidityCategory", "IngestionTimeUTC",
}
assert required.issubset(weather_df.columns)
assert len(weather_df) == len(CITIES)
assert weather_df["City"].is_unique
assert not weather_df[list(required)].isnull().any().any()
assert weather_df["Latitude"].between(-90, 90).all()
assert weather_df["Longitude"].between(-180, 180).all()
assert weather_df["HumidityPct"].between(0, 100).all()
weather_df.sort_values("City").reset_index(drop=True)

## What comes next

A separate weather-forecasting repository should collect historical observations on a schedule, store them with an idempotent key such as `(city, observation_time)`, create time-based train/validation/test splits, compare against seasonal baselines, and only then train forecasting models. Current observations from one notebook run are not sufficient training data.